# Imports

In [ ]:
!pip install torch torchvision

# === Imports ===
import os, json, math, random, time, glob, copy
from typing import Dict, List, Tuple
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files, drive
import zipfile

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import (
    Dataset, DataLoader, Subset, ConcatDataset, WeightedRandomSampler
)
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)


# Load in Datasets

## Unzip from Drive (and Check)

In [ ]:
drive.mount('/content/drive')

zip_path1 = '/content/drive/MyDrive/D-Fire.zip'
zip_path2 = '/content/drive/MyDrive/Fire Detection.v1i.yolov11.zip'
#zip_path3 = '/content/drive/MyDrive/Real Fire-Detection.zip'
zip_path4 = '/content/drive/MyDrive/SDXL Realistic Dataset.zip' # FILL IN WHEN YOU KNOW
zip_path5 = '/content/drive/MyDrive/SDXL Unrealistic Dataset.zip' # FILL IN WHEN YOU KNOW
zip_path6 = '/content/drive/MyDrive/SDXL New Unrealistic Dataset 1.zip' # FILL IN WHEN YOU KNOW
zip_path7 = '/content/drive/MyDrive/Fire Overlay Pngs.zip' # FILL IN WHEN YOU KNOW

# Real Fires

with zipfile.ZipFile(zip_path1, 'r') as zip_ref:
    zip_ref.extractall('/content/d-fire')

with zipfile.ZipFile(zip_path2, 'r') as zip_ref:
    zip_ref.extractall('/content/val-fire')

#with zipfile.ZipFile(zip_path3, 'r') as zip_ref:
    #zip_ref.extractall('/content/small-fire')

with zipfile.ZipFile(zip_path4, 'r') as zip_ref:
    zip_ref.extractall('/content/sdxl-realistic')

with zipfile.ZipFile(zip_path5, 'r') as zip_ref:
    zip_ref.extractall('/content/sdxl-unrealistic')

with zipfile.ZipFile(zip_path6, 'r') as zip_ref:
    zip_ref.extractall('/content/sdxl-new-unrealistic')

with zipfile.ZipFile(zip_path7, 'r') as zip_ref:
    zip_ref.extractall('/content/fire-overlays')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Create the Datasets

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

### Synthetic Fire Datasets

In [ ]:
synth_realistic = ImageFolder(
    root="/content/sdxl-realistic/SDXL Realistic Dataset",
    transform=transform
)


synth_unrealistic = ImageFolder(
    root="/content/sdxl-unrealistic/SDXL Unrealistic Dataset",
    transform=transform
)


synth_new_unrealistic = ImageFolder(
    root="/content/sdxl-new-unrealistic/SDXL New Unrealistic Dataset 1",
    transform=transform
)

fire_overlays = ImageFolder(
    root="/content/fire-overlays/Fire Overlay Pngs",
    transform=transform
)


### D Fire Dataset

In [ ]:
class DFireDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_paths = sorted([
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.endswith((".jpg", ".png"))
        ])
        self.label_paths = sorted([
            os.path.join(label_dir, f)
            for f in os.listdir(label_dir)
            if f.endswith(".txt")
        ])
        self.transform = transform

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)

        label_path = self.label_paths[idx]
        class_id = 0  # default: no fire

        try:
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) > 0 and parts[0] == '1':  # class_id 1 = fire
                        class_id = 1  # fire present
                        break

        except:
            print(f"!! Could not read label file: {label_path}")

        return image, class_id

dfire_train = DFireDataset(
    image_dir='/content/d-fire/train/images',
    label_dir='/content/d-fire/train/labels',
    transform=transform)

dfire_val = DFireDataset(
    image_dir='/content/d-fire/test/images',
    label_dir='/content/d-fire/test/labels',
    transform=transform)

### Test Datasets

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class FireDataset(Dataset):
    """
    Returns (image_tensor, label, img_name)
      - label = 1 if any line in the YOLO .txt has the target class_id
      - label = 0 otherwise
    Assumes images are in image_dir and labels are in label_dir with matching stems.
    """
    def __init__(self, image_dir, label_dir, fire_class_id=0, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.fire_class_id = str(fire_class_id)
        self.transform = transform


        self.image_paths = sorted([
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.endswith((".jpg", ".png"))
        ])

        # build the corresponding label paths by stem
        self.label_paths = [
            os.path.join(label_dir, os.path.splitext(os.path.basename(p))[0] + ".txt")
            for p in self.image_paths
        ]

        if len(self.image_paths) == 0:
            raise RuntimeError(f"No images found in {image_dir}")

    def __getitem__(self, idx):
        img_path  = self.image_paths[idx]
        label_path = self.label_paths[idx]
        img_name = os.path.basename(img_path)

        # load image
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # default: no fire
        y = 0
        try:
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    if parts[0] == self.fire_class_id:
                        y = 1
                        break
        except FileNotFoundError:
            # no label file -> keep y = 0
            pass
        except Exception as e:
            print(f"!! Could not read label file: {label_path} ({e})")

        return image, y, img_name

real_test_clean = FireDataset(
    image_dir='/content/val-fire/train/images',
    label_dir='/content/val-fire/train/labels',
    fire_class_id=1,
    transform=transform
)



# Functions

In [ ]:
from typing import Tuple, Dict, Optional
import numpy as np
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score, confusion_matrix
)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_model(num_classes=2):
    m = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_feats = m.fc.in_features
    m.fc = nn.Linear(in_feats, num_classes)
    return m

@torch.no_grad()
def collect_probs(model, loader, device) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    model.eval()
    y_true, p_pred, paths = [], [], []
    for batch in loader:
        if len(batch) == 3:
            x, y, ps = batch
            paths.extend(ps)
        else:
            x, y = batch
            ps = None
        x = x.to(device, non_blocking=True)
        logits = model(x)
        p = torch.softmax(logits, dim=1)[:, 1]
        p_pred.append(p.detach().cpu().numpy())
        y_true.append(y.numpy())
    y_true = np.concatenate(y_true).astype(np.int32)
    p_pred = np.concatenate(p_pred).astype(np.float32)
    paths = np.array(paths) if len(paths) else None
    return y_true, p_pred, paths

def find_tau_max_f1(y_true: np.ndarray, p_pred: np.ndarray) -> float:

    prec, rec, thr = precision_recall_curve(y_true, p_pred)
    f1 = 2*prec*rec/(prec+rec+1e-9)
    i = int(np.nanargmax(f1)) #index of best f1
    i_thr = max(0, min(i-1, len(thr)-1))
    return float(thr[i_thr])

def summarise_metrics(y_true: np.ndarray, p_pred: np.ndarray, tau: float) -> Dict:

    # threshold-free
    auroc = roc_auc_score(y_true, p_pred)
    auprc = average_precision_score(y_true, p_pred)
    prevalence = float(y_true.mean())
    print(f"prevalence: {prevalence}")
    ap_lift = float(auprc - prevalence)

    # thresholded
    y_hat = (p_pred >= tau).astype(int)
    acc  = accuracy_score(y_true, y_hat)
    prec = precision_score(y_true, y_hat, zero_division=0)
    rec  = recall_score(y_true, y_hat, zero_division=0)
    f1   = 2*prec*rec/(prec+rec+1e-9)

    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    tpr = tp / (tp + fn) if (tp+fn)>0 else 0.0
    tnr = tn / (tn + fp) if (tn+fp)>0 else 0.0
    bal_acc = 0.5 * (tpr + tnr)

    return {
        "acc": acc, "precision": prec, "recall": rec, "f1": f1,
        "auprc": auprc, "auroc": auroc,
        "prevalence": prevalence, "ap_lift": ap_lift, "bal_acc": bal_acc,
        "n": int(len(y_true)),
    }

#for checkpoint selection
def val_ap_lift_score(model, loader, device) -> float:
    y_true, p_pred, _ = collect_probs(model, loader, device)
    auprc = average_precision_score(y_true, p_pred)
    prevalence = float(y_true.mean())
    return float(auprc - prevalence)



In [ ]:
def run_resnet_experiment(
    run_id: str,
    seed: int,
    train_dataset: torch.utils.data.Dataset,
    val_dataset: torch.utils.data.Dataset,
    test_dataset: torch.utils.data.Dataset,
    batch_size: int = 64,
    max_epochs: int = 10,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    use_mixed_precision: bool = True,
    num_workers: int = 0,
    pin_memory: bool = True,
    save_dir: Optional[str] = None,
) -> Dict:
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model().to(device)

    # dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=use_mixed_precision)

    best_state = None
    best_score = -float("inf")
    used_epoch = 0

    # train and pick best by val AP-lift
    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch in train_loader:
            xb, yb = batch[:2]
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_mixed_precision):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        score = val_ap_lift_score(model, val_loader, device)
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            used_epoch = epoch

    if best_state is not None:
        model.load_state_dict(best_state)

    # tau* from validation (max F1)
    yv, pv, _ = collect_probs(model, val_loader, device)
    tau_star = find_tau_max_f1(yv, pv)

    # test probs + paths
    yt, pt, paths = collect_probs(model, test_loader, device)

    # metrics at tau* and 0.5
    m_tau = summarise_metrics(yt, pt, tau_star)
    m_05  = summarise_metrics(yt, pt, 0.5)

    # save artifacts
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(save_dir, f"{run_id}_seed{seed}_best.pt"))
        np.savez(os.path.join(save_dir, f"{run_id}_seed{seed}_clean.npz"),
                 y_true=yt, p_pred=pt, paths=paths, tau_star=tau_star, seed=seed)

    # return summary
    return {
        "run_id": run_id,
        "seed": seed,
        "trained_epochs": used_epoch,
        "test_auroc": m_tau["auroc"],
        "test_auprc": m_tau["auprc"],
        "test_ap_lift": m_tau["ap_lift"],
        "n_test": m_tau["n"],
        "acc@tau*": m_tau["acc"],
        "precision@tau*": m_tau["precision"],
        "recall@tau*": m_tau["recall"],
        "f1@tau*": m_tau["f1"],
        "bal_acc@tau*": m_tau["bal_acc"],
        "acc@0.5": m_05["acc"],
        "precision@0.5": m_05["precision"],
        "recall@0.5": m_05["recall"],
        "f1@0.5": m_05["f1"],
        "bal_acc@0.5": m_05["bal_acc"],
    }


In [ ]:
def run_resnet_baseline(
    run_id: str,
    seed: int,
    val_dataset: torch.utils.data.Dataset,
    test_dataset: torch.utils.data.Dataset,
    batch_size: int = 64,
    num_workers: int = 0,
    pin_memory: bool = True,
    save_dir: Optional[str] = None,
) -> Dict:
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model().to(device)

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )

    yv, pv, _ = collect_probs(model, val_loader, device)
    tau_star = find_tau_max_f1(yv, pv)

    yt, pt, paths = collect_probs(model, test_loader, device)

    m_tau = summarise_metrics(yt, pt, tau_star)
    m_05  = summarise_metrics(yt, pt, 0.5)

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(save_dir, f"{run_id}_seed{seed}_best.pt"))
        np.savez(os.path.join(save_dir, f"{run_id}_seed{seed}_clean.npz"),
                 y_true=yt, p_pred=pt, paths=paths, tau_star=tau_star, seed=seed)

    return {
        "run_id": run_id,
        "seed": seed,
        "trained_epochs": 0,
        "test_auroc": m_tau["auroc"],
        "test_auprc": m_tau["auprc"],
        "test_ap_lift": m_tau["ap_lift"],
        "n_test": m_tau["n"],
        "acc@tau*": m_tau["acc"],
        "precision@tau*": m_tau["precision"],
        "recall@tau*": m_tau["recall"],
        "f1@tau*": m_tau["f1"],
        "bal_acc@tau*": m_tau["bal_acc"],
        "acc@0.5": m_05["acc"],
        "precision@0.5": m_05["precision"],
        "recall@0.5": m_05["recall"],
        "f1@0.5": m_05["f1"],
        "bal_acc@0.5": m_05["bal_acc"],
    }


# Stage 1

In [ ]:
SEEDS = list(range(30, 40))

In [ ]:
import os, time, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

RUNS = [
    ("Synthetic_Overlays", fire_overlays)
]

# Setup save dirs
SAVE_ROOT = "/content/drive/MyDrive/fire_proj/outputs_stage1"
os.makedirs(SAVE_ROOT, exist_ok=True)

SESSION_TAG = time.strftime("%Y%m%d-%H%M%S")
SESSION_DIR = os.path.join(SAVE_ROOT, f"session_{SESSION_TAG}")
os.makedirs(SESSION_DIR, exist_ok=True)


results_rows = []

# train/eval on synthetic overlays
for seed in SEEDS:
    print(f"\n=== SEED {seed} ===")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    run_id, train_ds = RUNS[0]
    print(f"Running {run_id} (clean)")
    clean_row = run_resnet_experiment(
        run_id=run_id,
        seed=seed,
        train_dataset=train_ds,
        val_dataset=dfire_val,
        test_dataset=real_test_clean,
        save_dir=SESSION_DIR,  # saves {run_id}_seed{seed}_best.pt and _clean.npz
    )
    clean_row_with_cond = clean_row.copy()
    clean_row_with_cond["condition"] = "clean"
    results_rows.append(clean_row_with_cond)
    print(clean_row_with_cond)

# save session
df_results = pd.DataFrame(results_rows)

cols_order = [
    "run_id","seed","trained_epochs","condition",
    "test_auroc","test_auprc","test_ap_lift","n_test",
    "acc@tau*","precision@tau*","recall@tau*","f1@tau*","bal_acc@tau*",
    "acc@0.5","precision@0.5","recall@0.5","f1@0.5","bal_acc@0.5",
]
df_results = df_results[cols_order]

session_csv = os.path.join(SESSION_DIR, "stage1_results.csv")
df_results.to_csv(session_csv, index=False)
print("\nSaved session CSV ->", session_csv)

MASTER_CSV = os.path.join(SAVE_ROOT, "stage1_results_master.csv")
if os.path.exists(MASTER_CSV):
    old = pd.read_csv(MASTER_CSV)
    combined = pd.concat([old, df_results], ignore_index=True)
    combined = (combined
                .sort_values(by=["run_id","seed","condition"])
                .drop_duplicates(subset=["run_id","seed","condition"], keep="last"))
else:
    combined = df_results.copy()

combined.to_csv(MASTER_CSV, index=False)
print("Updated master CSV ->", MASTER_CSV)


In [ ]:
import os, glob, time, pandas as pd
from IPython.display import display

SAVE_ROOT = "/content/drive/MyDrive/fire_proj/outputs_stage1"
ART_IDX   = os.path.join(SAVE_ROOT, "artifacts_index.csv")

npz_paths  = glob.glob(os.path.join(SESSION_DIR, "*.npz"))
ckpt_paths = glob.glob(os.path.join(SESSION_DIR, "*_best.pt"))

def parse_npz(path):
    # "{run_id}_seed{seed}_{condition}.npz"
    base = os.path.basename(path)[:-4]
    run_id, tail = base.split("_seed", 1)
    seed_str, condition = tail.split("_", 1)
    return run_id, int(seed_str), condition

def parse_ckpt(path):
    # "{run_id}_seed{seed}_best.pt"
    base = os.path.basename(path)[:-3]
    run_id, tail = base.split("_seed", 1)
    seed_str = tail.rsplit("_best", 1)[0]
    return run_id, int(seed_str)

# Map (run_id, seed) -> ckpt path
ckpt_map = { parse_ckpt(p): p for p in ckpt_paths }

timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
rows = []
for p in npz_paths:
    rid, sd, cond = parse_npz(p)
    rows.append({
        "run_id": rid,
        "seed": sd,
        "condition": cond,
        "npz_path": p,
        "ckpt_path": ckpt_map.get((rid, sd), ""),
        "session_dir": SESSION_DIR,
        "timestamp": timestamp,
    })

new_idx = pd.DataFrame(rows)

if os.path.exists(ART_IDX):
    all_idx = pd.concat([pd.read_csv(ART_IDX), new_idx], ignore_index=True)
    all_idx = (all_idx
               .sort_values(["run_id","seed","condition","timestamp"])
               .drop_duplicates(subset=["run_id","seed","condition"], keep="last"))
else:
    all_idx = new_idx

all_idx.to_csv(ART_IDX, index=False)
print("Updated artifacts index ->", ART_IDX)
display(all_idx.sort_values(["run_id","seed","condition"]))


# Stage 2

In [ ]:
import numpy as np
from torch.utils.data import Subset, ConcatDataset

def indices_by_label(ds, pos_label=1):
    pos, neg = [], []
    for i in range(len(ds)):
        y = int(ds[i][1])  # assumes (x, y, ...)
        (pos if y == pos_label else neg).append(i)
    return pos, neg

def sample_fixed(items, n, rng, replace=False):
    items = np.array(items)
    if n == "all":
        return items.tolist()
    n = int(n)
    if n > len(items) and not replace:
        raise ValueError(f"Asked for {n} but only {len(items)} available (replace=False).")
    idx = rng.choice(len(items), size=n, replace=replace)
    return items[idx].tolist()

def build_stage2_sets_from_synth(
    dfire_train,                 # D-Fire TRAIN split
    synth_realistic,             # synthetic dataset (with fire + no fire)
    synth_unrealistic,           # synthetic dataset (with fire + no fire)
    n_real_pos=1390,             # how many REAL fire images
    n_real_neg=1390,             # how many REAL no-fire images
    data_seed=42
):
    rng = np.random.RandomState(data_seed)

    # --- Real slices ---
    real_pos_idx, real_neg_idx = indices_by_label(dfire_train, pos_label=1)
    real_pos_sel = sample_fixed(real_pos_idx, n_real_pos, rng, replace=False)
    real_neg_sel = sample_fixed(real_neg_idx, n_real_neg, rng, replace=False)
    real_pos_ds  = Subset(dfire_train, real_pos_sel)
    real_neg_ds  = Subset(dfire_train, real_neg_sel)
    real_only    = ConcatDataset([real_pos_ds, real_neg_ds])

    # --- Synthetic (use ALL images, no sampling) ---
    synthR_ds = synth_realistic
    synthU_ds = synth_unrealistic

    # --- Arms ---
    synthR_plus_real = ConcatDataset([synthR_ds, real_only])
    synthU_plus_real = ConcatDataset([synthU_ds, real_only])

    return real_only, synthR_plus_real, synthU_plus_real



In [ ]:
results = []

for n_real in [500, 1000]:

    real_only, synthR_plus_real, synth_Uoverlays_plus_real = build_stage2_sets_from_synth(
        dfire_train=dfire_train,
        synth_realistic=synth_realistic,
        synth_unrealistic=fire_overlays,
        n_real_pos=n_real,
        n_real_neg=n_real,
        n_synth="all",
        data_seed=42)


    for seed in SEEDS:

        row = run_resnet_experiment("Plus_Synthetic_Overlays", seed, synth_Uoverlays_plus_real, dfire_val, real_test)
        row["n_real"] = n_real
        results.append(row)


        print(f"results.append({row})")


        row = run_resnet_experiment("Plus_Synthetic_Realistic", seed, synthR_plus_real, dfire_val, real_test)
        row["n_real"] = n_real
        results.append(row)



        print(f"results.append({row})")



# References

https://medium.com/we-talk-data/expert-guide-to-training-models-with-pytorchs-imagenet-dataset-927b69f80a76

https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html